# Six ways to split training across GPUs, checked on small numbers

One GPU runs out of room in one of three ways: the model's weights do not fit, the stored activations do not fit, or the work takes too long. Each method below splits a different thing across GPUs. Each cell does the split by hand on a few numbers, with every "GPU" a slice of a tensor and every message between GPUs an ordinary assignment, and checks the answer against one device.

Runs on any CPU in seconds (fp64, so "the same" means the same to 15 digits). `guide.md` Part 10 explains each one from scratch; `parallel_checks.py` holds the code.

In [1]:
import os, sys
if not os.path.exists("parallel_checks.py"):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/nikshrimali/a13-reversibility.git"], check=True)
    os.chdir("a13-reversibility")
sys.path.insert(0, os.getcwd())
import parallel_checks as pc

## 1. Data parallelism: split the batch

Every GPU has the whole model and a different slice of the examples. Each computes the gradient of its own slice, then they average (an **all-reduce**). Because the loss is an average over examples, the average of the two halves' gradients is exactly the full batch's gradient. Saves time, not memory: each GPU still holds the whole model. a12 shows how ZeRO removes that duplication.

In [2]:
pc.data_parallel();


== Data parallelism: split the batch ==
  gradient on one device:
 tensor([[ 6.0000, -3.6875],
        [ 5.3750, -2.1250]], dtype=torch.float64)
  GPU 0's gradient (examples 0,1):
 tensor([[ 7.0000, -5.8750],
        [ 5.2500, -2.3750]], dtype=torch.float64)
  GPU 1's gradient (examples 2,3):
 tensor([[ 5.0000, -1.5000],
        [ 5.5000, -1.8750]], dtype=torch.float64)
  their average:
 tensor([[ 6.0000, -3.6875],
        [ 5.3750, -2.1250]], dtype=torch.float64)
  average vs one device: largest difference 0.0e+00 -> SAME
  each GPU holds: the whole model (16 bytes) and half the examples; sends its whole gradient (16 bytes) to be averaged, every step


## 2. Tensor parallelism: split one layer's weights

The MLP's first matrix is cut by columns: each GPU makes half the hidden numbers on its own, and GELU works number by number, so it needs nothing from the other GPU. The second matrix is cut by rows: each GPU multiplies its half and gets a partial output, and one all-reduce adds the two partials. One message per layer, but it happens in every layer, forward and backward, so it needs a fast link between the GPUs.

In [3]:
pc.tensor_parallel();


== Tensor parallelism: split one layer's weights ==
  x: [1.0, 2.0]  W1: [[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]]  W2: [[0.1, 0.2], [0.3, 0.4], [0.5, 0.6], [0.7, 0.8]]
  GPU 0 hidden (columns 0,1): [0.9508, 1.2869]   GPU 1 hidden (columns 2,3): [1.6242, 1.9545]
  GPU 0 partial output: [0.4812, 0.7049]   GPU 1 partial output: [2.1803, 2.5381]
  sum: [2.6614, 3.2431]   one device: [2.6614, 3.2431]
  split MLP vs one device: largest difference 4.4e-16 -> SAME
  each GPU holds half of W1 and W2 (32 of 64 bytes); sends its partial output (8 bytes per token) once per layer, forward and backward


## 3. Pipeline parallelism: split the layers

GPU 0 holds layers 0-1, GPU 1 holds layers 2-3, like two stations of an assembly line in different buildings. Only the activations between the two stations travel. The catch is waiting: GPU 1 has nothing to do until GPU 0 finishes the first piece. Cutting the batch into micro-batches keeps both busy most of the time; the idle share (the **bubble**) shrinks as the number of micro-batches grows.

In [4]:
pc.pipeline_parallel(stages=2, micro=4);
pc.pipeline_parallel(stages=2, micro=16);


== Pipeline parallelism: split the layers ==
  pipelined output vs one device: largest difference 0.0e+00 -> SAME
  forward schedule, 2 GPUs x 4 micro-batches (number = micro-batch, . = idle):
    GPU 0: 0 1 2 3 .
    GPU 1: . 0 1 2 3
  idle share (the 'bubble') = (GPUs - 1) / (micro-batches + GPUs - 1) = 1/5 = 20%
  each GPU holds 2 of 4 layers; GPU 0 sent 96 bytes of activations this step

== Pipeline parallelism: split the layers ==
  pipelined output vs one device: largest difference 0.0e+00 -> SAME
  forward schedule, 2 GPUs x 16 micro-batches (number = micro-batch, . = idle):
    GPU 0: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 .
    GPU 1: . 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  idle share (the 'bubble') = (GPUs - 1) / (micro-batches + GPUs - 1) = 1/17 = 6%
  each GPU holds 2 of 4 layers; GPU 0 sent 96 bytes of activations this step


## 4. Sequence parallelism: split the sentence for the token-by-token parts

LayerNorm (and dropout) treat each token on its own, so each GPU can process its own half of the sentence and store only that half's activations. It is used together with tensor parallelism, for the parts of the layer that tensor parallelism cannot split.

In [5]:
pc.sequence_parallel();


== Sequence parallelism: split the sentence for the token-by-token parts ==
  token 0 by hand: mean = (1+2+3+6)/4 = 3, spread = sqrt(((1-3)^2+(2-3)^2+(3-3)^2+(6-3)^2)/4) = sqrt(3.5)
    -> (-2, -1, 0, 3) / 1.8708 = [-1.069, -0.5345, 0.0, 1.6036]
  two halves vs whole sentence: largest difference 0.0e+00 -> SAME
  each GPU stores the LayerNorm's saved input for half the tokens (32 of 64 bytes); attention still needs every token, so the halves are gathered before it and split again after


## 5. Context parallelism: split attention itself along the sentence

Attention is the part where every token looks at every earlier token, so it cannot simply be cut in half. Each GPU keeps the queries for its own tokens; the keys and values travel round a ring of GPUs, one chunk at a time. Softmax needs all scores of a row before it can divide, so each GPU keeps a running maximum and running totals and rescales them when a chunk with a bigger score arrives (the **online softmax**). No GPU ever builds the whole table of scores, which is what makes very long sentences possible.

In [6]:
pc.context_parallel(T=8, D=2, gpus=2);
pc.context_parallel(T=12, D=4, gpus=3);


== Context parallelism: split attention itself along the sentence ==
  ring attention vs one device: largest difference 1.1e-16 -> SAME
  each GPU holds queries, keys and values for 4 of 8 tokens, and never builds the full 8 x 8 score table; key/value bytes passed round the ring: 64

== Context parallelism: split attention itself along the sentence ==
  ring attention vs one device: largest difference 3.3e-16 -> SAME
  each GPU holds queries, keys and values for 4 of 12 tokens, and never builds the full 12 x 12 score table; key/value bytes passed round the ring: 384


## 6. Expert parallelism: each expert on its own GPU

In a mixture of experts (a14), a router sends each token to one expert. Put each expert on its own GPU, and send each token to the GPU of its expert and back (an **all-to-all**). Each GPU holds only its experts' weights.

In [7]:
pc.expert_parallel();


== Expert parallelism: each expert on its own GPU ==
  router picks expert [0, 1, 0, 0] for tokens 0..3
  experts on separate GPUs vs one device: largest difference 0.0e+00 -> SAME
  each GPU holds one expert (16 of 32 bytes of expert weights); tokens sent there and back: 64 bytes


## All six at once

In [8]:
res = pc.main()


== Data parallelism: split the batch ==
  gradient on one device:
 tensor([[ 6.0000, -3.6875],
        [ 5.3750, -2.1250]], dtype=torch.float64)
  GPU 0's gradient (examples 0,1):
 tensor([[ 7.0000, -5.8750],
        [ 5.2500, -2.3750]], dtype=torch.float64)
  GPU 1's gradient (examples 2,3):
 tensor([[ 5.0000, -1.5000],
        [ 5.5000, -1.8750]], dtype=torch.float64)
  their average:
 tensor([[ 6.0000, -3.6875],
        [ 5.3750, -2.1250]], dtype=torch.float64)
  average vs one device: largest difference 0.0e+00 -> SAME
  each GPU holds: the whole model (16 bytes) and half the examples; sends its whole gradient (16 bytes) to be averaged, every step

== Tensor parallelism: split one layer's weights ==
  x: [1.0, 2.0]  W1: [[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]]  W2: [[0.1, 0.2], [0.3, 0.4], [0.5, 0.6], [0.7, 0.8]]
  GPU 0 hidden (columns 0,1): [0.9508, 1.2869]   GPU 1 hidden (columns 2,3): [1.6242, 1.9545]
  GPU 0 partial output: [0.4812, 0.7049]   GPU 1 partial output: [2.1803